# CS229 — Moduł 1: Regresja liniowa — Notebook (część programistyczna)

**Zakres:** Rozdział 1 `main_notes` — implementujemy **od zera** (bez gotowych bibliotek ML):
`predict` / `cost`, **batch** i **stochastic gradient descent**, **równanie normalne** (normal
equation) oraz **locally weighted linear regression (LWR)**.

**Jak pracować:**
1. Uruchom komórki z importami i danymi.
2. Wypełnij każdą komórkę oznaczoną `# TODO` (rdzeń algorytmu — bez `sklearn` itp.).
3. Uruchom **komórkę sprawdzającą** — asercje + wykresy powiedzą Ci, czy działa.
4. Sekcja **Rozwiązania** na końcu — zajrzyj dopiero po próbie.

> ⚠️ Notebook był testowany wykonaniem na wersji z poprawnymi rozwiązaniami (asercje przechodzą,
> wykresy się generują). Jeśli komórka sprawdzająca rzuca `NotImplementedError`, to znaczy, że
> dany `TODO` nie jest jeszcze wypełniony.


## Importy i dane syntetyczne

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(229)

# --- Dane syntetyczne: jedna cecha (feature), by łatwo rysować ---
n = 80
x_raw = rng.uniform(0.0, 10.0, size=n)
true_theta = np.array([2.0, 1.3])               # [intercept, slope] w skali surowej
y = true_theta[0] + true_theta[1] * x_raw + rng.normal(0.0, 1.0, size=n)

# Standaryzacja cechy: poprawia uwarunkowanie (conditioning) -> stabilny, szybki gradient descent.
x_mean, x_std = x_raw.mean(), x_raw.std()
x_norm = (x_raw - x_mean) / x_std

# Macierz projektowa (design matrix) z wyrazem wolnym (intercept) x0 = 1.
X = np.column_stack([np.ones(n), x_norm])        # shape (n, 2)

print("X shape:", X.shape, "| y shape:", y.shape)
print("Dla danych standaryzowanych X^T X powinno być ~ n * I:")
print(np.round(X.T @ X, 3))


## Część 1 — Hipoteza i koszt

Hipoteza: $h_\theta(x)=\theta^T x$ (wektorowo: `X @ theta`).
Koszt: $J(\theta)=\tfrac12\sum_{i}\big(h_\theta(x^{(i)})-y^{(i)}\big)^2$.

In [ ]:
def predict(X, theta):
    """Hipoteza h_theta(X) = X @ theta. Zwraca wektor predykcji kształtu (n,)."""
    # TODO: zwróć X @ theta
    raise NotImplementedError("TODO: predict")


def cost(X, y, theta):
    """J(theta) = 1/2 * sum_i (h(x_i) - y_i)^2  -> skalar."""
    # TODO: policz residua (X@theta - y) i zwróć 0.5 * sum(residua**2)
    raise NotImplementedError("TODO: cost")


## Część 2 — Batch gradient descent

Gradient: $\nabla_\theta J(\theta)=X^T(X\theta-\vec{y})$. Aktualizacja:
$\theta:=\theta-\alpha\,\nabla_\theta J(\theta)$.

In [ ]:
def compute_gradient(X, y, theta):
    """nabla_theta J = X^T (X theta - y) -> wektor (d,)."""
    # TODO
    raise NotImplementedError("TODO: compute_gradient")


def batch_gradient_descent(X, y, alpha, num_iters):
    """Batch GD. Zwraca (theta, history), gdzie history[t] = J(theta) po iteracji t."""
    theta = np.zeros(X.shape[1])
    history = []
    for _ in range(num_iters):
        # TODO: 1) grad = compute_gradient(...), 2) theta = theta - alpha*grad,
        #       3) history.append(cost(...))
        raise NotImplementedError("TODO: batch_gradient_descent")
    return theta, np.array(history)


## Część 3 — Stochastic gradient descent

Jedna aktualizacja **na pojedynczy przykład**:
$\theta:=\theta-\alpha\big(x^{(i)T}\theta-y^{(i)}\big)x^{(i)}$. W każdej epoce przechodzimy
przez wszystkie przykłady w losowej kolejności.

In [ ]:
def stochastic_gradient_descent(X, y, alpha, num_epochs, seed=0):
    """SGD. Zwraca (theta, history), gdzie history[e] = J(theta) po epoce e."""
    rng_local = np.random.default_rng(seed)
    n_, d = X.shape
    theta = np.zeros(d)
    history = []
    for _ in range(num_epochs):
        order = rng_local.permutation(n_)
        for i in order:
            xi, yi = X[i], y[i]
            # TODO: err = xi @ theta - yi ; theta = theta - alpha * err * xi
            raise NotImplementedError("TODO: stochastic_gradient_descent")
        history.append(cost(X, y, theta))
    return theta, np.array(history)


## Część 4 — Równanie normalne (normal equation)

Rozwiązanie zamknięte: $\theta=(X^TX)^{-1}X^T\vec{y}$.
W praktyce **nie** odwracaj macierzy jawnie — użyj `np.linalg.solve(A, b)` na układzie
$X^TX\,\theta=X^T\vec{y}$ (stabilniej numerycznie).

In [ ]:
def normal_equation(X, y):
    """theta = (X^T X)^{-1} X^T y. Użyj np.linalg.solve(X.T@X, X.T@y)."""
    # TODO
    raise NotImplementedError("TODO: normal_equation")


## Część 5 — Locally weighted linear regression (LWR)

Dla punktu zapytania $x$ (query) ważymy przykłady
$w^{(i)}=\exp\!\big(-\lVert x^{(i)}-x\rVert^2/(2\tau^2)\big)$ (liczone na cechach bez interceptu),
i rozwiązujemy **ważone** równanie normalne $\theta=(X^TWX)^{-1}X^TW\vec{y}$, $W=\mathrm{diag}(w)$.
Predykcja w punkcie: $x^T\theta$. (Wyprowadzenie wzoru: zad. 2.1 w zestawie matematycznym.)

In [ ]:
def lwr_predict(X, y, x_query, tau):
    """
    X: (n,d) design matrix (z interceptem), y: (n,), x_query: (d,) (z interceptem), tau: bandwidth.
    Zwraca skalar = predykcja w x_query.
    """
    # TODO:
    # 1) diffs = X[:, 1:] - x_query[1:]      # odległości liczone bez kolumny interceptu
    # 2) dist_sq = np.sum(diffs**2, axis=1)
    # 3) w = np.exp(-dist_sq / (2*tau**2)) ;  W = np.diag(w)
    # 4) theta = np.linalg.solve(X.T @ W @ X, X.T @ W @ y)
    # 5) return x_query @ theta
    raise NotImplementedError("TODO: lwr_predict")


## ✅ Komórka sprawdzająca (asercje + wykresy)

Uruchom **po** wypełnieniu wszystkich `TODO`. Jeśli wszystko gra — zobaczysz „Wszystkie asercje
przeszły” i trzy wykresy.

In [ ]:
# 1) Równanie normalne odzyskuje prawdziwe parametry (z dokładnością do szumu).
theta_ne = normal_equation(X, y)
# theta_ne jest w skali standaryzowanej; przeliczamy na surową dla interpretacji:
slope_raw = theta_ne[1] / x_std
intercept_raw = theta_ne[0] - theta_ne[1] * x_mean / x_std
print(f"Normal eq (skala surowa): intercept={intercept_raw:.3f}, slope={slope_raw:.3f} "
      f"| prawdziwe: {true_theta[0]}, {true_theta[1]}")
assert np.allclose([intercept_raw, slope_raw], true_theta, atol=0.5), "Normal eq za daleko od prawdy"

# 2) Batch GD: koszt maleje monotonicznie i zbiega do rozwiązania zamkniętego.
theta_bgd, hist_bgd = batch_gradient_descent(X, y, alpha=0.01, num_iters=2000)
assert hist_bgd[-1] < hist_bgd[0], "Koszt BGD nie zmalał"
assert np.all(np.diff(hist_bgd) <= 1e-6), "Koszt BGD nie jest monotonicznie nierosnący"
assert np.allclose(theta_bgd, theta_ne, atol=1e-3), "BGD nie zbiega do rozwiązania zamkniętego"

# 3) SGD trafia w pobliże optimum (koszt bliski minimalnemu J).
theta_sgd, hist_sgd = stochastic_gradient_descent(X, y, alpha=0.005, num_epochs=100, seed=1)
J_min = cost(X, y, theta_ne)
assert cost(X, y, theta_sgd) <= 1.2 * J_min + 1e-6, "SGD nie zbliżył się do minimum"

# 4) LWR: granica dużego tau ~ zwykła regresja (OLS).
x_q = np.array([1.0, 0.5])
assert np.isclose(lwr_predict(X, y, x_q, tau=1e6), x_q @ theta_ne, atol=1e-3), "LWR(tau->inf) != OLS"

# 5) LWR: mniejsze tau => elastyczniejsze dopasowanie => mniejszy błąd treningowy.
def _lwr_train_mse(tau):
    preds = np.array([lwr_predict(X, y, X[i], tau) for i in range(len(y))])
    return np.mean((preds - y) ** 2)
assert _lwr_train_mse(0.3) < _lwr_train_mse(50.0), "Małe tau nie dało elastyczniejszego dopasowania"

print("\n✅ Wszystkie asercje przeszły.")

# ---------------------- Wykresy ----------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# (a) zbieżność kosztu: BGD vs SGD
axes[0].plot(hist_bgd, label="batch GD")
axes[0].plot(hist_sgd, label="SGD (po epoce)")
axes[0].axhline(J_min, ls="--", color="k", lw=1, label="min J (normal eq)")
axes[0].set_yscale("log"); axes[0].set_xlabel("iteracja / epoka"); axes[0].set_ylabel("J(theta)")
axes[0].set_title("Zbieżność kosztu"); axes[0].legend()

# (b) dopasowanie liniowe
order = np.argsort(x_raw)
axes[1].scatter(x_raw, y, s=18, alpha=0.7, label="dane")
axes[1].plot(x_raw[order], (X @ theta_ne)[order], color="red", lw=2, label="regresja liniowa")
axes[1].set_xlabel("x"); axes[1].set_ylabel("y"); axes[1].set_title("Dopasowanie (normal eq)")
axes[1].legend()

# (c) LWR dla różnych tau
xs = np.linspace(x_raw.min(), x_raw.max(), 200)
xs_norm = (xs - x_mean) / x_std
axes[2].scatter(x_raw, y, s=18, alpha=0.4, label="dane")
for tau in [0.3, 0.7, 2.0]:
    preds = np.array([lwr_predict(X, y, np.array([1.0, xq]), tau) for xq in xs_norm])
    axes[2].plot(xs, preds, lw=1.8, label=f"LWR tau={tau}")
axes[2].set_xlabel("x"); axes[2].set_ylabel("y"); axes[2].set_title("LWR: wpływ bandwidth tau")
axes[2].legend()

plt.tight_layout()
plt.savefig("/home/claude/m1_wykresy.png", dpi=80)
plt.show()
print("Wykresy wygenerowane.")


---
## 📕 Rozwiązania (zajrzyj dopiero po próbie)

Poniżej kompletne, przetestowane implementacje. Możesz je odkomentować/skopiować, jeśli utkniesz.


In [ ]:
# ROZWIĄZANIA — referencyjne implementacje (te same, które przechodzą asercje).
#
# def predict(X, theta):
#     return X @ theta
#
# def cost(X, y, theta):
#     resid = predict(X, theta) - y
#     return 0.5 * np.sum(resid ** 2)
#
# def compute_gradient(X, y, theta):
#     return X.T @ (predict(X, theta) - y)
#
# def batch_gradient_descent(X, y, alpha, num_iters):
#     theta = np.zeros(X.shape[1]); history = []
#     for _ in range(num_iters):
#         theta = theta - alpha * compute_gradient(X, y, theta)
#         history.append(cost(X, y, theta))
#     return theta, np.array(history)
#
# def stochastic_gradient_descent(X, y, alpha, num_epochs, seed=0):
#     rng_local = np.random.default_rng(seed); n_, d = X.shape
#     theta = np.zeros(d); history = []
#     for _ in range(num_epochs):
#         for i in rng_local.permutation(n_):
#             theta = theta - alpha * (X[i] @ theta - y[i]) * X[i]
#         history.append(cost(X, y, theta))
#     return theta, np.array(history)
#
# def normal_equation(X, y):
#     return np.linalg.solve(X.T @ X, X.T @ y)
#
# def lwr_predict(X, y, x_query, tau):
#     dist_sq = np.sum((X[:, 1:] - x_query[1:]) ** 2, axis=1)
#     W = np.diag(np.exp(-dist_sq / (2.0 * tau ** 2)))
#     theta = np.linalg.solve(X.T @ W @ X, X.T @ W @ y)
#     return x_query @ theta
print("Sekcja rozwiązań — odkomentuj w razie potrzeby.")
